In [0]:
%pip install python-dotenv
#dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv

notebook_path = "/Workspace/Users/nascimento.germano@gmail.com/estagio-empregadados-turma-2"
env_path = os.path.join(notebook_path, ".env")
load_dotenv(dotenv_path=env_path)
load_dotenv()

SECRET_SCOPE = "squad3"

def obter_config(chave):
    valor = os.getenv(chave)
    if valor:
        return valor
    try:
        return dbutils.secrets.get(scope=SECRET_SCOPE, key=chave)
    except Exception:
        return None

CLIENT_ID       = obter_config("CLIENT_ID")
CLIENT_SECRET   = obter_config("CLIENT_SECRET")
TENANT_ID       = obter_config("TENANT_ID")
STORAGE_ACCOUNT = obter_config("STORAGE_ACCOUNT")
CONTAINER       = obter_config("CONTAINER")

if not all([CLIENT_ID, CLIENT_SECRET, TENANT_ID, STORAGE_ACCOUNT, CONTAINER]):
    raise ValueError(
        f"❌ ERRO: configuração incompleta.\n"
        f"Caminho do .env tentado: {env_path} | Secret scope: {SECRET_SCOPE}"
    )

OAUTH_ENDPOINT = f"https://login.microsoftonline.com/{TENANT_ID}/oauth2/token"

RAW_BASE_PATH    = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net"
BRONZE_BASE_PATH = f"{RAW_BASE_PATH}/bronze"

BRONZE_ITENS_VENDA_PATH = f"{BRONZE_BASE_PATH}/squad3/itens_venda"
BRONZE_VENDAS_PATH      = f"{BRONZE_BASE_PATH}/squad3/vendas"

print("✅ Configuração carregada com sucesso!")

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# O ambiente Serverless não permite spark.conf.set para fs.azure.*:
# a autenticação OAuth (Service Principal) é passada como opção em cada leitura/escrita.
ADLS_OPTIONS = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": CLIENT_ID,
    "fs.azure.account.oauth2.client.secret": CLIENT_SECRET,
    "fs.azure.account.oauth2.client.endpoint": OAUTH_ENDPOINT,
}

def ler_csv_adls(caminho):
    return (spark.read.format("csv").options(**ADLS_OPTIONS)
        .option("header", "true").option("inferSchema", "true")
        .load(caminho)
        .withColumn("bronze_source_file", F.col("_metadata.file_path")))

def ler_delta(caminho):
    return spark.read.format("delta").options(**ADLS_OPTIONS).load(caminho)

def delta_existe(caminho):
    """True se existe _delta_log no caminho. Outros erros (auth, rede) são propagados."""
    try:
        (spark.read.format("binaryFile").options(**ADLS_OPTIONS)
            .option("pathGlobFilter", "*.json")
            .load(f"{caminho}/_delta_log")
            .columns)  # força a análise do plano (Spark Connect é lazy)
        return True
    except Exception as e:
        marcadores = ["Path does not exist", "PATH_NOT_FOUND", "does not exist", "doesn't exist"]
        if any(m in str(e) for m in marcadores):
            return False
        raise

print("✅ Funções criadas.")

In [0]:
SILVER_BASE_PATH        = f"{RAW_BASE_PATH}/silver"
SILVER_ITENS_VENDA_PATH = f"{SILVER_BASE_PATH}/squad3/itens_venda"
SILVER_REJEITADOS_PATH  = f"{SILVER_BASE_PATH}/squad3/itens_venda_rejeitados"
RAW_PERECIVEIS_PATH     = f"{RAW_BASE_PATH}/batch-data/physical_produtos_pereciveis.csv"

print("✅ Caminhos da Silver definidos.")

In [0]:
print("🔎 Lendo a Bronze (Delta)...")

ausentes = [p for p in (BRONZE_ITENS_VENDA_PATH, BRONZE_VENDAS_PATH) if not delta_existe(p)]
if ausentes:
    raise FileNotFoundError(
        "❌ Bronze não encontrada em: " + ", ".join(ausentes) +
        "\nExecute o notebook de ingestão Bronze antes da Silver."
    )

df_bronze_itens  = ler_delta(BRONZE_ITENS_VENDA_PATH)
df_bronze_vendas = ler_delta(BRONZE_VENDAS_PATH)

print(f"Itens na Bronze  : {df_bronze_itens.count()}")
print(f"Vendas na Bronze : {df_bronze_vendas.count()}")
display(df_bronze_itens.limit(10))

In [0]:
df_vendas_ref = (
    df_bronze_vendas
    .groupBy("id_transacao")
    .agg(F.min("dt_venda").alias("dt_venda"),
         F.min("id_loja").alias("id_loja"))
)
print("✅ Dados de vendas preparados para o enriquecimento.")

In [0]:
df_pereciveis = ler_csv_adls(RAW_PERECIVEIS_PATH)
print("✅ Cadastro de produtos perecíveis carregado.")
df_pereciveis.printSchema()
display(df_pereciveis.limit(20))

In [0]:
df_pereciveis_ref = (
    df_pereciveis
    .select(F.col("sku").cast("string").alias("sku_perecivel"),
            F.lower(F.trim(F.col("unidade_medida"))).alias("unidade_medida_produto"))
    .groupBy("sku_perecivel")
    .agg(F.min("unidade_medida_produto").alias("unidade_medida_produto"))
    .withColumn("eh_perecivel", F.lit(True))
)
display(df_pereciveis_ref.limit(20))

In [0]:
COLS_QUARENTENA = ["id_item_venda", "id_transacao", "codigo_barras_produto",
                   "quantidade", "preco_unitario_registro", "valor_total_item",
                   "bronze_source_file"]

def para_quarentena(df, motivo):
    """Padroniza registros rejeitados (tudo string) com o motivo da rejeição."""
    motivo_col = F.lit(motivo) if isinstance(motivo, str) else motivo
    return (df.withColumn("motivo_rejeicao", motivo_col)
              .select([F.col(c).cast("string").alias(c) for c in COLS_QUARENTENA] +
                      [F.col("motivo_rejeicao").cast("string")])
              .withColumn("quarentena_at", F.current_timestamp()))

In [0]:
# PK nula → quarentena
df_pk_nula = df_bronze_itens.filter(F.col("id_item_venda").isNull())

# Duplicados na Bronze (append): mantém o mais recente
w = Window.partitionBy("id_item_venda").orderBy(F.col("bronze_ingested_at").desc())
df_base = (df_bronze_itens
    .filter(F.col("id_item_venda").isNotNull())
    .withColumn("_rn", F.row_number().over(w))
    .filter("_rn = 1").drop("_rn"))

# Incremental: não reprocessa o que já está na Silver
if delta_existe(SILVER_ITENS_VENDA_PATH):
    ids_silver = ler_delta(SILVER_ITENS_VENDA_PATH).select("id_item_venda")
    df_base = df_base.join(ids_silver, "id_item_venda", "left_anti")
    print("↪️ Silver existente: apenas itens novos serão processados.")
else:
    print("🆕 Silver ainda não existe (primeira execução).")

In [0]:
df_orfaos = df_base.join(df_vendas_ref, on="id_transacao", how="left_anti")
df_silver = df_base.join(df_vendas_ref, on="id_transacao", how="inner")

print("✅ df_silver criado.")
df_silver.printSchema()
display(df_silver.limit(10))

In [0]:
df_silver = (
    df_silver
    .join(df_pereciveis_ref,
          F.col("codigo_barras_produto").cast("string") == F.col("sku_perecivel"),
          how="left")
    .withColumn("tipo_produto",
                F.when(F.col("eh_perecivel"), F.lit("PERECIVEL")).otherwise(F.lit("SECO")))
    # Itens SECO não constam no cadastro de perecíveis: tratados como unidade
    .withColumn("unidade_medida_produto",
                F.when(F.col("tipo_produto") == "SECO", F.lit("un"))
                 .otherwise(F.col("unidade_medida_produto")))
    .drop("sku_perecivel", "eh_perecivel")
)

print("✅ Produtos classificados.")
display(df_silver.select("codigo_barras_produto", "unidade_medida_produto", "tipo_produto").limit(20))

In [0]:
df_silver = (
    df_silver
    .withColumn("quantidade", F.col("quantidade").cast("DECIMAL(18,3)"))
    .withColumn(
        "quantidade_valida",
        F.when(F.col("quantidade").isNull() | (F.col("quantidade") <= 0), F.lit(False))
         .when(F.col("unidade_medida_produto").isin("kg", "l"), F.lit(True))
         .when(F.col("unidade_medida_produto") == "un",
               F.col("quantidade") == F.floor(F.col("quantidade")))
         .otherwise(F.lit(True))
    )
)
print("✅ Regra de quantidade aplicada (flag quantidade_valida).")

In [0]:
df_silver = (
    df_silver
    .withColumn("preco_unitario_registro", F.col("preco_unitario_registro").cast("DECIMAL(10,2)"))
    .withColumn("preco_valido",
                F.coalesce(F.col("preco_unitario_registro") > 0, F.lit(False)))
)
print("✅ Regra de preço aplicada (flag preco_valido).")

In [0]:
df_silver = (
    df_silver
    .withColumn("valor_total_item", F.col("valor_total_item").cast("DECIMAL(10,2)"))
    .withColumn("valor_total_calculado",
                F.round(F.col("preco_unitario_registro") * F.col("quantidade"), 2))
    .withColumn(
        "valor_total_consistente",
        F.coalesce(
            F.abs(F.col("valor_total_item") - F.col("valor_total_calculado")) <= F.lit(0.01),
            F.lit(False))
    )
)
print("✅ Consistência de valor total avaliada.")

In [0]:
from datetime import date, timedelta

def pascoa(ano):
    """Domingo de Páscoa (algoritmo gregoriano)."""
    a = ano % 19; b = ano // 100; c = ano % 100
    d = b // 4; e = b % 4; f = (b + 8) // 25; g = (b - f + 1) // 3
    h = (19*a + b - d - g + 15) % 30
    i = c // 4; k = c % 4
    l = (32 + 2*e + 2*i - h - k) % 7
    m = (a + 11*h + 22*l) // 451
    mes = (h + l - 7*m + 114) // 31
    dia = ((h + l - 7*m + 114) % 31) + 1
    return date(ano, mes, dia)

FERIADOS_FIXOS = [
    (1, 1,  "Confraternização Universal"),
    (4, 21, "Tiradentes"),
    (5, 1,  "Dia Mundial do Trabalho"),
    (9, 7,  "Independência do Brasil"),
    (10, 12, "Nossa Senhora Aparecida"),
    (11, 2,  "Finados"),
    (11, 15, "Proclamação da República"),
    (12, 25, "Natal"),
]

feriados = {}
for ano in range(2000, date.today().year + 2):
    for mes, dia, nome in FERIADOS_FIXOS:
        feriados[date(ano, mes, dia)] = nome
    if ano >= 2024:  # Lei 14.759/2023
        feriados[date(ano, 11, 20)] = "Dia Nacional de Zumbi e da Consciência Negra"
    feriados[pascoa(ano) - timedelta(days=2)] = "Sexta-feira Santa"

df_feriados = spark.createDataFrame(
    [(d, n) for d, n in feriados.items()],
    "data_feriado date, nome_feriado string")
display(df_feriados.orderBy("data_feriado").limit(20))

In [0]:
df_silver = (
    df_silver
    .withColumn("data_venda", F.to_date("dt_venda"))
    .join(F.broadcast(df_feriados), F.col("data_venda") == F.col("data_feriado"), "left")
    .withColumn("venda_em_feriado", F.col("data_feriado").isNotNull())
    .drop("data_venda", "data_feriado", "nome_feriado")
)

print("✅ Flag venda_em_feriado criada.")
display(df_silver.select("id_item_venda", "dt_venda", "venda_em_feriado").limit(20))

In [0]:
df_silver = df_silver.withColumn("silver_processed_at", F.current_timestamp())
print("✅ silver_processed_at adicionado.")

In [0]:
df_silver = df_silver.withColumn(
    "motivo_rejeicao",
    F.concat_ws(";",
        F.when(~F.col("quantidade_valida"),        F.lit("QUANTIDADE_INVALIDA")),
        F.when(~F.col("preco_valido"),             F.lit("PRECO_INVALIDO")),
        F.when(~F.col("valor_total_consistente"),  F.lit("VALOR_TOTAL_INCONSISTENTE")))
)

df_rej_regras = df_silver.filter(F.col("motivo_rejeicao") != "")
df_silver     = df_silver.filter(F.col("motivo_rejeicao") == "")

df_quarentena = (
    para_quarentena(df_pk_nula, "PK_NULA")
    .unionByName(para_quarentena(df_orfaos, "FK_ORFA"))
    .unionByName(para_quarentena(df_rej_regras, F.col("motivo_rejeicao")))
)
print("✅ Regras aplicadas: válidos → Silver | inválidos → quarentena.")

In [0]:
df_silver = (
    df_silver
    .withColumn("ano_venda", F.coalesce(F.year("dt_venda"),  F.lit(0)))
    .withColumn("mes_venda", F.coalesce(F.month("dt_venda"), F.lit(0)))
    .select(
        "id_item_venda", "id_transacao", "id_loja", "codigo_barras_produto",
        "quantidade", "preco_unitario_registro", "valor_total_item", "dt_venda",
        "unidade_medida_produto", "tipo_produto", "venda_em_feriado",
        "silver_processed_at", "ano_venda", "mes_venda"
    )
)

print("✅ Estrutura final da Silver definida.")
df_silver.printSchema()

In [0]:
def soma(cond):
    return F.coalesce(F.sum(cond.cast("int")), F.lit(0))

m = df_silver.agg(
    F.count("*").alias("registros"),
    soma(F.col("id_item_venda").isNull()).alias("pk_nula"),
    F.countDistinct("id_item_venda").alias("pk_distintas"),
    soma(F.col("tipo_produto") == "PERECIVEL").alias("pereciveis"),
    soma(F.col("id_transacao").isNull()).alias("fk_nula"),
    soma(F.col("quantidade").isNull() | (F.col("quantidade") <= 0)).alias("qtd_invalida"),
    soma(F.col("preco_unitario_registro").isNull() |
         (F.col("preco_unitario_registro") <= 0)).alias("preco_invalido"),
    soma(F.col("venda_em_feriado")).alias("em_feriado"),
).first()

pk_duplicada = m["registros"] - m["pk_nula"] - m["pk_distintas"]

print(f"Registros (a gravar)  : {m['registros']}")
print(f"Itens perecíveis      : {m['pereciveis']}")
print(f"PK nula               : {m['pk_nula']}")
print(f"PK duplicada          : {pk_duplicada}")
print(f"FK nula               : {m['fk_nula']}")
print(f"Quantidade inválida   : {m['qtd_invalida']}")
print(f"Preço inválido        : {m['preco_invalido']}")
print(f"Vendas em feriado     : {m['em_feriado']}")

print("\n🚧 Quarentena por motivo:")
display(df_quarentena.groupBy("motivo_rejeicao").count().orderBy(F.desc("count")))

if any([m["pk_nula"], pk_duplicada, m["fk_nula"], m["qtd_invalida"], m["preco_invalido"]]):
    raise ValueError("❌ Silver viola regras de qualidade. Gravação interrompida.")

display(df_silver.orderBy("dt_venda").limit(20))

In [0]:
(df_quarentena.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .options(**ADLS_OPTIONS)
    .save(SILVER_REJEITADOS_PATH))
print("🚧 Quarentena gravada.")

(df_silver.write
    .format("delta")
    .mode("append")
    .options(**ADLS_OPTIONS)
    .partitionBy("ano_venda", "mes_venda")
    .save(SILVER_ITENS_VENDA_PATH))
print("✅ Silver gravada em Delta com sucesso.")

In [0]:
df_silver_check = ler_delta(SILVER_ITENS_VENDA_PATH)

print("✅ Silver Delta lida novamente com sucesso.")
df_silver_check.printSchema()
display(df_silver_check.limit(10))
display(df_silver_check.groupBy("ano_venda", "mes_venda").count()
        .orderBy("ano_venda", "mes_venda"))